In [1]:
import json
ex_list = []
with open('examples.txt', encoding='utf-8') as f:
    lines = f.readlines()
    for line in lines:
        ex_list.append(json.loads(line.strip()))
print(ex_list[0]['word'])

walk


In [5]:
def writeData(ex, f):
    res = json.dumps({
        "level": ex['level'],
        "example": ex['example'],
        "translation": ex['translation'],
        "pronunciation": ex['pronunciation'],
    }, ensure_ascii=False)
    data = {"messages":[{"role":"system","content": 'Given English word and Korean meaning, provide the following fields in a JSON dict, where applicable: "level" (Level of the word from 1 to 10), "example" (Natural English example sentence of the word), "translation" (Korean translation of the example), "pronunciation" (The pronunciation of the word in Hangul). Highlighting the word and meaning in the example and translation respectively with *. If the word and meaning are not matched, awkward, or wrong, provide an empty JSON dict.'},{"role":"user", "content": f"word: {ex['word']}, meaning: {ex['meaning']}"},{"role":"assistant","content": res }]}
    f.write(json.dumps(data, ensure_ascii = False) + '\n')

In [6]:
with open('voca.jsonl', 'w+', encoding='utf-8') as f:
    for ex in ex_list:
        writeData(ex, f)

In [7]:
with open('train.jsonl', 'w+', encoding='utf-8') as f:
    for ex in ex_list[:50]:
        writeData(ex, f)

In [8]:
with open('valid.jsonl', 'w+', encoding='utf-8') as f:
    for ex in ex_list[50:]:
        writeData(ex, f)

In [4]:
import json
import tiktoken # for token counting
import numpy as np
from collections import defaultdict

In [5]:
data_path = "./train.jsonl"

# Load the dataset
with open(data_path, 'r', encoding='utf-8') as f:
    dataset = [json.loads(line) for line in f]

# Initial dataset stats
print("Num examples:", len(dataset))
print("First example:")
for message in dataset[0]["messages"]:
    print(message)

Num examples: 70
First example:
{'role': 'developer', 'content': 'Given English word and Korean meaning, provide the following fields in a JSON dict, where applicable: "level" (Level of the word from 1 to 10), "example" (Natural English example sentence of the word), "translation" (Korean translation of the example). Highlighting the word and meaning in the example and translation respectively with *. If the word and meaning are not matched, awkward, or wrong, provide an empty JSON dict.'}
{'role': 'user', 'content': 'word: walk, meaning: 걷다'}
{'role': 'assistant', 'content': '{"level": 1, "example": "She likes to *walk* in the park every morning.", "translation": "그녀는 매일 아침 공원에서 *걷는* 것을 좋아합니다."}'}


In [6]:
# Format error checks
format_errors = defaultdict(int)

for ex in dataset:
    if not isinstance(ex, dict):
        format_errors["data_type"] += 1
        continue
        
    messages = ex.get("messages", None)
    if not messages:
        format_errors["missing_messages_list"] += 1
        continue
        
    for message in messages:
        if "role" not in message or "content" not in message:
            format_errors["message_missing_key"] += 1
        
        if any(k not in ("role", "content", "name", "function_call", "weight") for k in message):
            format_errors["message_unrecognized_key"] += 1
        
        if message.get("role", None) not in ("developer", "user", "assistant", "function"):
            format_errors["unrecognized_role"] += 1
            
        content = message.get("content", None)
        function_call = message.get("function_call", None)
        
        if (not content and not function_call) or not isinstance(content, str):
            format_errors["missing_content"] += 1
    
    if not any(message.get("role", None) == "assistant" for message in messages):
        format_errors["example_missing_assistant_message"] += 1

if format_errors:
    print("Found errors:")
    for k, v in format_errors.items():
        print(f"{k}: {v}")
else:
    print("No errors found")

No errors found
